# Avaliacao de Modelos — Credit Risk Intelligence Platform

Este notebook realiza a avaliacao rigorosa dos modelos treinados no notebook 16_gold_training, utilizando um conjunto de validacao estratificado (holdout 20%) derivado de `credit_risk.gold.ml_train`.

**Importante**: Nao utiliza ml_test (sem TARGET). Nao faz CV, tuning, SMOTE, PCA ou feature selection. Nao registra no MLflow.

In [0]:
# ============================================================
# INSTALACAO DE BIBLIOTECAS ADICIONAIS (XGBoost E LightGBM)
# ============================================================
%pip install xgboost lightgbm -q

In [0]:
# ============================================================
# SECAO 1 — CONFIGURACAO E IMPORTS
# ============================================================
import warnings
import time
import uuid
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, recall_score,
    precision_score, f1_score, accuracy_score,
    confusion_matrix, roc_curve, precision_recall_curve,
    brier_score_loss, balanced_accuracy_score
)
from sklearn.calibration import calibration_curve

# Tentar importar XGBoost e LightGBM
xgb_available = False
lgb_available = False
try:
    from xgboost import XGBClassifier
    xgb_available = True
except ImportError:
    print("WARNING: XGBoost nao disponivel")
try:
    from lightgbm import LGBMClassifier
    lgb_available = True
except ImportError:
    print("WARNING: LightGBM nao disponivel")

# Configuracoes globais
warnings.filterwarnings('ignore')
RANDOM_STATE = 42
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Caminhos centralizados
TRAIN_TABLE = "credit_risk.gold.ml_train"
TEST_TABLE = "credit_risk.gold.ml_test"
TRAINING_RESULTS_TABLE = "credit_risk.analytics.model_training_results"
FEATURE_IMPORTANCE_TABLE = "credit_risk.analytics.feature_importance_training"
ANALYTICS_SCHEMA = "credit_risk.analytics"

# ID de execucao para auditoria
EXECUTION_ID = str(uuid.uuid4())
START_TIME = time.time()
START_TIMESTAMP = datetime.now()

# Dicionarios globais
all_eval_results = []
trained_models = {}
predictions_store = {}

print(f"17_EVALUATION - Credit Risk Intelligence Platform")
print(f"Execution ID: {EXECUTION_ID}")
print(f"Start Time: {START_TIMESTAMP}")
print(f"Random State: {RANDOM_STATE}")

In [0]:
# ============================================================
# SECAO 2 — CARREGAMENTO DOS DADOS
# ============================================================
print("=" * 60)
print("SECAO 2 — Carregamento dos Dados")
print("=" * 60)

# Carregar ml_train
print("Carregando credit_risk.gold.ml_train...")
df_train = spark.table(TRAIN_TABLE).toPandas()

# Carregar resultados do treinamento (notebook 16)
print("Carregando credit_risk.analytics.model_training_results...")
df_training_results = spark.table(TRAINING_RESULTS_TABLE).toPandas()

# Carregar feature importance do treinamento
print("Carregando credit_risk.analytics.feature_importance_training...")
df_feature_importance = spark.table(FEATURE_IMPORTANCE_TABLE).toPandas()

# Validacoes
print(f"\n--- Validacoes ---")
print(f"ml_train: {df_train.shape[0]:,} rows, {df_train.shape[1]} cols")
print(f"TARGET presente: {'Sim' if 'TARGET' in df_train.columns else 'Nao - ERRO'}")
print(f"SK_ID_CURR presente: {'Sim' if 'SK_ID_CURR' in df_train.columns else 'Nao'}")
n_duplicates = df_train['SK_ID_CURR'].duplicated().sum()
print(f"Duplicidade SK_ID_CURR: {n_duplicates}")

# Resumo
print(f"\n--- Resumo ---")
print(f"Rows: {df_train.shape[0]:,}")
print(f"Features (excluindo SK_ID_CURR e TARGET): {df_train.shape[1] - 2}")
print(f"\nTARGET distribution:")
target_dist = df_train['TARGET'].value_counts().sort_index()
for val, cnt in target_dist.items():
    pct = cnt / len(df_train) * 100
    print(f"  TARGET={val}: {cnt:,} ({pct:.2f}%)")
class_imbalance = target_dist[0] / target_dist[1]
print(f"Class imbalance ratio: {class_imbalance:.2f}:1")
total_nulls = df_train.isnull().sum().sum()
print(f"\nTotal nulls: {total_nulls:,}")

# Training results summary
print(f"\n--- Resultados do Notebook 16 ---")
print(df_training_results[['model_name', 'roc_auc', 'pr_auc', 'cv_mean']].to_string(index=False))
print("\nCarregamento concluido!")

In [0]:
# ============================================================
# SECOES 3 e 4 — DIVISAO DE VALIDACAO E IDENTIFICACAO DE FEATURES
# ============================================================
print("=" * 60)
print("SECAO 3 — Divisao de Validacao (Stratified Split 80/20)")
print("=" * 60)

# Separar features e target
exclude_cols = ['SK_ID_CURR', 'TARGET']
feature_cols = [c for c in df_train.columns if c not in exclude_cols]

X = df_train[feature_cols].copy()
y = df_train['TARGET'].copy()
sk_ids = df_train['SK_ID_CURR'].copy()

print(f"X (features): {X.shape}")
print(f"y (target): {y.shape}")
print(f"Features: {len(feature_cols)}")

# Verificar exclusoes
assert 'TARGET' not in X.columns, "ERRO: TARGET nas features!"
assert 'SK_ID_CURR' not in X.columns, "ERRO: SK_ID_CURR nas features!"

# Stratified split 80/20 (mesmo random_state do notebook 16)
X_tr, X_val, y_tr, y_val, ids_tr, ids_val = train_test_split(
    X, y, sk_ids,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print(f"\n--- Estrategia de Validacao ---")
print(f"Train interno:        {X_tr.shape[0]:,} registros ({X_tr.shape[0]/len(X)*100:.1f}%)")
print(f"Validation (holdout): {X_val.shape[0]:,} registros ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Random state: {RANDOM_STATE}")
print(f"Stratification: Sim (preserva distribuicao do TARGET)")
print(f"Data leakage prevention: SK_ID_CURR excluido das features")

# Distribuicao das classes
print(f"\n--- Distribuicao das Classes ---")
for y_set, name in [(y_tr, "Train"), (y_val, "Validation")]:
    for val, cnt in y_set.value_counts().sort_index().items():
        pct = cnt / len(y_set) * 100
        print(f"  {name} TARGET={val}: {cnt:,} ({pct:.2f}%)")

# Identificar colunas numericas e categoricas
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"\n--- Identificacao de Features ---")
print(f"ID: SK_ID_CURR (nao usado como feature)")
print(f"TARGET: TARGET (nao usado como feature)")
print(f"Features numericas: {len(numeric_features)}")
print(f"Features categoricas: {len(categorical_features)}")
print(f"Total de features: {len(feature_cols)}")

# scale_pos_weight para modelos boosting
n_pos = int(y_tr.sum())
n_neg = int(len(y_tr) - n_pos)
scale_pos_weight = n_neg / n_pos
print(f"\nscale_pos_weight: {scale_pos_weight:.4f} (pos={n_pos:,}, neg={n_neg:,})")

# Preprocessor (mesma config do notebook 16)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
)
print("\nConfiguracao concluida!")

In [0]:
# ============================================================
# SECAO 5 — RECONSTRUCAO DOS MODELOS
# ============================================================
print("=" * 60)
print("SECAO 5 — Reconstructed dos Modelos")
print("=" * 60)

# Mesma configuracao do notebook 16_gold_training
model_configs = {
    'Dummy (most_frequent)': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE))
    ]),
    'Dummy (stratified)': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', DummyClassifier(strategy='stratified', random_state=RANDOM_STATE))
    ]),
    'Logistic Regression': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(
            class_weight='balanced',
            max_iter=2000,
            solver='lbfgs',
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),
    'Random Forest': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=200,
            max_depth=15,
            min_samples_leaf=50,
            class_weight='balanced',
            n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ]),
}

if xgb_available:
    model_configs['XGBoost'] = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', XGBClassifier(
            scale_pos_weight=scale_pos_weight,
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            eval_metric='logloss'
        ))
    ])
else:
    print("WARNING: XGBoost nao disponivel")

if lgb_available:
    model_configs['LightGBM'] = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', LGBMClassifier(
            scale_pos_weight=scale_pos_weight,
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1
        ))
    ])
else:
    print("WARNING: LightGBM nao disponivel")

print(f"\nModelos a serem treinados: {list(model_configs.keys())}")
print(f"Total: {len(model_configs)} modelos")
print("\nTreinando modelos no subconjunto TRAIN...")

for model_name, pipeline in model_configs.items():
    print(f"\n--- Treinando: {model_name} ---")
    t0 = time.time()
    pipeline.fit(X_tr, y_tr)
    train_time = time.time() - t0
    trained_models[model_name] = pipeline
    print(f"  Tempo: {train_time:.1f}s")

print(f"\n{len(trained_models)} modelos treinados com sucesso!")
print("Importante: Sem CV, sem tuning, sem SMOTE, sem PCA, sem feature selection.")

In [0]:
# ============================================================
# SECAO 6 — PREDICOES
# ============================================================
print("=" * 60)
print("SECAO 6 — Predicoes")
print("=" * 60)

all_predictions = []

for model_name, pipeline in trained_models.items():
    print(f"Gerando predicoes: {model_name}...")

    y_pred = pipeline.predict(X_val)

    # Probabilidades (classe positiva)
    if hasattr(pipeline, 'predict_proba'):
        y_proba = pipeline.predict_proba(X_val)[:, 1]
    elif hasattr(pipeline, 'decision_function'):
        y_proba = pipeline.decision_function(X_val)
    else:
        y_proba = y_pred.astype(float)

    # Armazenar
    predictions_store[model_name] = {
        'y_pred': y_pred,
        'y_proba': y_proba
    }

    # Tabela de predicoes com SK_ID_CURR para rastreabilidade
    pred_df = pd.DataFrame({
        'SK_ID_CURR': ids_val.values,
        'TARGET': y_val.values,
        'PREDICTION': y_pred,
        'PROBABILITY': y_proba,
        'MODEL': model_name
    })
    all_predictions.append(pred_df)

predictions_df = pd.concat(all_predictions, ignore_index=True)
print(f"\nTotal de predicoes: {len(predictions_df):,}")
print(f"Modelos: {predictions_df['MODEL'].nunique()}")
print(f"Registros por modelo: {len(y_val):,}")
print(f"\nAmostra de predicoes:")
print(predictions_df.head(10).to_string(index=False))
print("\nPredicoes concluidas!")

In [0]:
# ============================================================
# SECAO 7 — METRICAS PRINCIPAIS
# ============================================================
print("=" * 60)
print("SECAO 7 — Metricas Principais (Threshold = 0.50)")
print("=" * 60)

THRESHOLD = 0.50
eval_results = []

for model_name in trained_models.keys():
    y_pred = predictions_store[model_name]['y_pred']
    y_proba = predictions_store[model_name]['y_proba']

    # Metricas
    roc = roc_auc_score(y_val, y_proba)
    pr = average_precision_score(y_val, y_proba)
    acc = accuracy_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred, zero_division=0)
    rec = recall_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred, zero_division=0)

    # Confusion matrix para specificity
    cm = confusion_matrix(y_val, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    balanced_acc = balanced_accuracy_score(y_val, y_pred)
    brier = brier_score_loss(y_val, y_proba)

    eval_results.append({
        'MODEL': model_name,
        'ROC_AUC': round(roc, 4),
        'PR_AUC': round(pr, 4),
        'ACCURACY': round(acc, 4),
        'PRECISION': round(prec, 4),
        'RECALL': round(rec, 4),
        'F1': round(f1, 4),
        'SPECIFICITY': round(specificity, 4),
        'BALANCED_ACCURACY': round(balanced_acc, 4),
        'BRIER_SCORE': round(brier, 4)
    })

metrics_df = pd.DataFrame(eval_results)
metrics_df = metrics_df.sort_values('ROC_AUC', ascending=False).reset_index(drop=True)

print("\n=== Tabela Comparativa de Metricas (Threshold = 0.50) ===\n")
print(metrics_df.to_string(index=False))

print(f"\n--- Trade-offs ---")
print("Recall: capacidade de detectar inadimplentes (TP / (TP + FN))")
print("Precision: proporcao de verdadeiros positivos entre os alertas (TP / (TP + FP))")
print("ROC-AUC: capacidade de ranking em todos os thresholds")
print("PR-AUC: capacidade de ranking considerando desbalanceamento")
print("Brier Score: calibracao das probabilidades (menor = melhor)")
print("Specificity: taxa de verdadeiros negativos (TN / (TN + FP))")
print("\nMetricas calculadas!")

In [0]:
# ============================================================
# SECAO 8 — MATRIZ DE CONFUSAO
# ============================================================
print("=" * 60)
print("SECAO 8 — Matriz de Confusao")
print("=" * 60)

confusion_results = []

for model_name in trained_models.keys():
    y_pred = predictions_store[model_name]['y_pred']
    cm = confusion_matrix(y_val, y_pred)
    tn, fp, fn, tp = cm.ravel()

    print(f"\n--- {model_name} ---")
    print(f"  TN={tn:,} | FP={fp:,} | FN={fn:,} | TP={tp:,}")
    print(f"  [{tn:>8,}  {fp:>8,}]")
    print(f"  [{fn:>8,}  {tp:>8,}]")

    confusion_results.append({
        'MODEL': model_name,
        'TN': int(tn), 'FP': int(fp),
        'FN': int(fn), 'TP': int(tp)
    })

confusion_df = pd.DataFrame(confusion_results)
print(f"\n=== Tabela de Matrizes de Confusao ===")
print(confusion_df.to_string(index=False))

# Visualizacao das matrizes de confusao
n_models = len(trained_models)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes_flat = axes.flatten()

for i, model_name in enumerate(trained_models.keys()):
    if i >= len(axes_flat):
        break
    y_pred = predictions_store[model_name]['y_pred']
    cm = confusion_matrix(y_val, y_pred)
    im = axes_flat[i].imshow(cm, cmap='Blues', aspect='auto')
    axes_flat[i].set_title(f'{model_name}', fontsize=11)
    axes_flat[i].set_xlabel('Predicted')
    axes_flat[i].set_ylabel('Actual')
    for r in range(cm.shape[0]):
        for c in range(cm.shape[1]):
            axes_flat[i].text(c, r, f'{cm[r,c]:,}', ha='center', va='center', fontsize=10)

# Esconder axes nao utilizados
for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle('Matrizes de Confusao por Modelo', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('/tmp/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nMatriz de confusao concluida!")

In [0]:
# ============================================================
# SECAO 9 — CURVA ROC
# ============================================================
print("=" * 60)
print("SECAO 9 — Curva ROC")
print("=" * 60)

fig, ax = plt.subplots(figsize=(10, 8))
roc_data_for_audit = []

for model_name in trained_models.keys():
    y_proba = predictions_store[model_name]['y_proba']
    fpr, tpr, thresholds = roc_curve(y_val, y_proba)
    roc_auc = roc_auc_score(y_val, y_proba)
    ax.plot(fpr, tpr, label=f'{model_name} (AUC={roc_auc:.4f})')
    # Salvar pontos para auditoria
    for f, t in zip(fpr, tpr):
        roc_data_for_audit.append({
            'MODEL': model_name, 'FPR': float(f), 'TPR': float(t), 'ROC_AUC': roc_auc
        })

# Linha de referencia (Random Classifier)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random Classifier (AUC=0.5)')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curve — Todos os Modelos', fontsize=13)
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

roc_audit_df = pd.DataFrame(roc_data_for_audit)
print(f"Pontos salvos para auditoria: {len(roc_audit_df):,}")
print("Curva ROC concluida!")

In [0]:
# ============================================================
# SECAO 10 — CURVA PRECISION-RECALL
# ============================================================
print("=" * 60)
print("SECAO 10 — Curva Precision-Recall")
print("=" * 60)

fig, ax = plt.subplots(figsize=(10, 8))
pr_data_for_audit = []

# Prevalencia da classe positiva (baseline)
prevalence = y_val.mean()

for model_name in trained_models.keys():
    y_proba = predictions_store[model_name]['y_proba']
    precision_arr, recall_arr, thresholds = precision_recall_curve(y_val, y_proba)
    pr_auc = average_precision_score(y_val, y_proba)
    ax.plot(recall_arr, precision_arr, label=f'{model_name} (AP={pr_auc:.4f})')
    # Salvar pontos para auditoria
    for p, r in zip(precision_arr, recall_arr):
        pr_data_for_audit.append({
            'MODEL': model_name, 'PRECISION': float(p), 'RECALL': float(r), 'PR_AUC': pr_auc
        })

# Linha de referencia (prevalencia da classe positiva)
ax.axhline(y=prevalence, color='k', linestyle='--', alpha=0.3, label=f'Prevalence ({prevalence:.4f})')
ax.set_xlabel('Recall', fontsize=11)
ax.set_ylabel('Precision', fontsize=11)
ax.set_title('Precision-Recall Curve — Todos os Modelos', fontsize=13)
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/pr_curve.png', dpi=150, bbox_inches='tight')
plt.show()

pr_audit_df = pd.DataFrame(pr_data_for_audit)
print(f"Pontos salvos para auditoria: {len(pr_audit_df):,}")
print("Curva Precision-Recall concluida!")

In [0]:
# ============================================================
# SECAO 11 — ANALISE DE THRESHOLD
# ============================================================
print("=" * 60)
print("SECAO 11 — Analise de Threshold")
print("=" * 60)

thresholds_list = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]
threshold_results = []

print(f"Thresholds avaliados: {thresholds_list}\n")

for model_name in trained_models.keys():
    print(f"--- {model_name} ---")
    y_proba = predictions_store[model_name]['y_proba']

    for thresh in thresholds_list:
        y_pred_t = (y_proba >= thresh).astype(int)
        cm = confusion_matrix(y_val, y_pred_t)
        tn, fp, fn, tp = cm.ravel()

        prec = precision_score(y_val, y_pred_t, zero_division=0)
        rec = recall_score(y_val, y_pred_t)
        f1_t = f1_score(y_val, y_pred_t, zero_division=0)
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        bal_acc = balanced_accuracy_score(y_val, y_pred_t)
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

        threshold_results.append({
            'MODEL': model_name,
            'THRESHOLD': thresh,
            'PRECISION': round(prec, 4),
            'RECALL': round(rec, 4),
            'F1': round(f1_t, 4),
            'SPECIFICITY': round(spec, 4),
            'BALANCED_ACCURACY': round(bal_acc, 4),
            'FPR': round(fpr, 4),
            'FNR': round(fnr, 4)
        })
    print()

threshold_df = pd.DataFrame(threshold_results)
print(f"=== Tabela de Threshold Analysis ({len(threshold_df)} registros) ===")
print(threshold_df.head(20).to_string(index=False))

# Graficos: Threshold x Metrica
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
metrics_to_plot = [('PRECISION', 'Precision'), ('RECALL', 'Recall'), ('F1', 'F1'), ('SPECIFICITY', 'Specificity')]
for ax, (col, title) in zip(axes.flatten(), metrics_to_plot):
    for model_name in trained_models.keys():
        model_data = threshold_df[threshold_df['MODEL'] == model_name]
        ax.plot(model_data['THRESHOLD'], model_data[col], marker='o', markersize=4, label=model_name)
    ax.set_xlabel('Threshold')
    ax.set_ylabel(title)
    ax.set_title(f'Threshold x {title}')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
plt.suptitle('Analise de Threshold por Modelo', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('/tmp/threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nAnalise de threshold concluida!")

In [0]:
# ============================================================
# SECAO 12 — ANALISE DE ERROS
# ============================================================
print("=" * 60)
print("SECAO 12 — Analise de Erros")
print("=" * 60)

error_records = []

for model_name in trained_models.keys():
    y_pred = predictions_store[model_name]['y_pred']
    y_proba = predictions_store[model_name]['y_proba']

    for i in range(len(y_val)):
        actual = int(y_val.iloc[i])
        pred = int(y_pred[i])
        proba = float(y_proba[i])
        sk_id = int(ids_val.iloc[i])

        if actual == 1 and pred == 1:
            error_type = 'TP'
        elif actual == 0 and pred == 0:
            error_type = 'TN'
        elif actual == 0 and pred == 1:
            error_type = 'FP'
        else:
            error_type = 'FN'

        error_records.append({
            'SK_ID_CURR': sk_id,
            'TARGET': actual,
            'PREDICTION': pred,
            'PROBABILITY': proba,
            'ERROR_TYPE': error_type,
            'MODEL': model_name
        })

errors_df = pd.DataFrame(error_records)

# Resumo por modelo
error_summary = errors_df.groupby(['MODEL', 'ERROR_TYPE']).size().unstack(fill_value=0)
error_summary = error_summary[['TP', 'TN', 'FP', 'FN']]
error_summary['FPR'] = (error_summary['FP'] / (error_summary['FP'] + error_summary['TN'])).round(4)
error_summary['FNR'] = (error_summary['FN'] / (error_summary['FN'] + error_summary['TP'])).round(4)

print("=== Resumo de Erros por Modelo ===")
print(error_summary.to_string())
print(f"\nTotal de registros de erro: {len(errors_df):,}")
print("\nAnalise de erros concluida!")

In [0]:
# ============================================================
# SECAO 13 — CALIBRACAO
# ============================================================
print("=" * 60)
print("SECAO 13 — Calibracao")
print("=" * 60)

N_BINS = 10
calibration_results = []

fig, ax = plt.subplots(figsize=(10, 8))

for model_name in trained_models.keys():
    y_proba = predictions_store[model_name]['y_proba']
    brier = brier_score_loss(y_val, y_proba)

    # Curva de calibracao
    frac_pos, mean_pred = calibration_curve(y_val, y_proba, n_bins=N_BINS, strategy='uniform')
    ax.plot(mean_pred, frac_pos, marker='o', label=f'{model_name} (Brier={brier:.4f})')

    # Dados por bin para tabela
    bin_edges = np.linspace(0, 1, N_BINS + 1)
    for b in range(N_BINS):
        mask = (y_proba >= bin_edges[b]) & (y_proba < bin_edges[b + 1])
        if b == N_BINS - 1:
            mask = (y_proba >= bin_edges[b]) & (y_proba <= bin_edges[b + 1])
        count = mask.sum()
        if count > 0:
            mean_p = float(y_proba[mask].mean())
            frac_p = float(y_val[mask].mean())
        else:
            mean_p = float((bin_edges[b] + bin_edges[b + 1]) / 2)
            frac_p = 0.0
        calibration_results.append({
            'MODEL': model_name,
            'BIN': f'{bin_edges[b]:.1f}-{bin_edges[b+1]:.1f}',
            'MEAN_PREDICTED_PROBABILITY': mean_p,
            'FRACTION_POSITIVE': frac_p,
            'COUNT': int(count)
        })

# Linha de referencia (perfeitamente calibrado)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Perfectly Calibrated')
ax.set_xlabel('Mean Predicted Probability', fontsize=11)
ax.set_ylabel('Fraction of Positives', fontsize=11)
ax.set_title('Calibration Curve — Todos os Modelos', fontsize=13)
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/calibration_curve.png', dpi=150, bbox_inches='tight')
plt.show()

calibration_df = pd.DataFrame(calibration_results)
print(f"\n=== Tabela de Calibracao ({len(calibration_df)} registros) ===")
print(calibration_df.head(15).to_string(index=False))
print("\nNota: Nenhuma calibracao foi aplicada aos modelos. Apenas medindo o comportamento atual.")
print("Calibracao concluida!")

In [0]:
# ============================================================
# SECAO 14 — FEATURE IMPORTANCE (DA TABELA DO NOTEBOOK 16)
# ============================================================
print("=" * 60)
print("SECAO 14 — Feature Importance (credit_risk.analytics.feature_importance_training)")
print("=" * 60)

# Modelos baseados em arvore que possuem feature importance registrada
tree_models = df_feature_importance['model_name'].unique().tolist()
print(f"Modelos com feature importance: {tree_models}")

fig, axes = plt.subplots(1, len(tree_models), figsize=(8 * len(tree_models), 8))
if len(tree_models) == 1:
    axes = [axes]

for i, model_name in enumerate(tree_models):
    fi_model = df_feature_importance[df_feature_importance['model_name'] == model_name].sort_values('ranking').head(20)

    print(f"\n--- {model_name} — Top 20 ---")
    print(fi_model[['feature', 'importance', 'ranking']].to_string(index=False))

    # Grafico de barras horizontal
    axes[i].barh(range(len(fi_model)), fi_model['importance'].values, color='steelblue')
    axes[i].set_yticks(range(len(fi_model)))
    axes[i].set_yticklabels(fi_model['feature'].values, fontsize=8)
    axes[i].invert_yaxis()
    axes[i].set_xlabel('Importance')
    axes[i].set_title(f'Top 20 Features — {model_name}', fontsize=11)

plt.suptitle('Feature Importance por Modelo (Top 20)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('/tmp/feature_importance_eval.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFeature importance concluida! Dados da tabela do notebook 16.")
print("Nenhuma feature foi alterada. Nenhum feature selection foi executado.")

In [0]:
# ============================================================
# SECAO 15 — COMPARACAO COM RESULTADOS DO 16_TRAINING
# ============================================================
print("=" * 60)
print("SECAO 15 — Comparacao com Resultados do 16_training")
print("=" * 60)

# Preparar dados de validacao (holdout)
val_metrics = metrics_df.set_index('MODEL')[['ROC_AUC', 'PR_AUC']].copy()

# Preparar dados de treinamento (notebook 16)
train_metrics = df_training_results.set_index('model_name')[['roc_auc', 'pr_auc', 'cv_mean']].copy()
train_metrics.columns = ['TRAINING_ROC_AUC', 'TRAINING_PR_AUC', 'CV_ROC_AUC']

# Alinhar indices
comparison = train_metrics.join(val_metrics.rename(columns={'ROC_AUC': 'VALIDATION_ROC_AUC', 'PR_AUC': 'VALIDATION_PR_AUC'}))

# Calcular deltas
comparison['ROC_AUC_DELTA'] = (comparison['VALIDATION_ROC_AUC'] - comparison['TRAINING_ROC_AUC']).round(4)
comparison['PR_AUC_DELTA'] = (comparison['VALIDATION_PR_AUC'] - comparison['TRAINING_PR_AUC']).round(4)

# Reordenar colunas
comparison = comparison[['TRAINING_ROC_AUC', 'VALIDATION_ROC_AUC', 'ROC_AUC_DELTA',
                         'TRAINING_PR_AUC', 'VALIDATION_PR_AUC', 'PR_AUC_DELTA', 'CV_ROC_AUC']]

print("\n=== Comparacao: Treinamento (16) vs Validacao (17) ===")
print(comparison.to_string())

# Sinalizar diferencas relevantes
print("\n--- Analise de Deltas ---")
for idx in comparison.index:
    roc_delta = comparison.loc[idx, 'ROC_AUC_DELTA']
    pr_delta = comparison.loc[idx, 'PR_AUC_DELTA']
    if abs(roc_delta) > 0.05:
        print(f"  {idx}: ROC-AUC delta = {roc_delta:+.4f} (DIFERENCA RELEVANTE)")
    elif abs(pr_delta) > 0.03:
        print(f"  {idx}: PR-AUC delta = {pr_delta:+.4f} (DIFERENCA RELEVANTE)")
    else:
        print(f"  {idx}: ROC-AUC delta = {roc_delta:+.4f}, PR-AUC delta = {pr_delta:+.4f}")

print("\nNota: Diferencas pequenas entre CV e validacao sao esperadas.")
print("Deltas grandes podem indicar overfitting, mas nao devem ser usados isoladamente.")
print("\nComparacao concluida!")

In [0]:
# ============================================================
# SECAO 16 — DATA LEAKAGE CHECK
# ============================================================
print("=" * 60)
print("SECAO 16 — Data Leakage Check")
print("=" * 60)

leakage_checks = []

# 1. SK_ID_CURR utilizado como feature?
sk_id_in_features = 'SK_ID_CURR' in feature_cols
leakage_checks.append({
    'CHECK': 'SK_ID_CURR utilizado como feature?',
    'RESULT': 'Sim' if sk_id_in_features else 'Nao',
    'STATUS': 'FAIL' if sk_id_in_features else 'PASS'
})

# 2. TARGET presente nas features?
target_in_features = 'TARGET' in feature_cols
leakage_checks.append({
    'CHECK': 'TARGET presente nas features?',
    'RESULT': 'Sim' if target_in_features else 'Nao',
    'STATUS': 'FAIL' if target_in_features else 'PASS'
})

# 3. Validation contaminando treinamento? (intersecao de SK_ID_CURR)
train_ids = set(ids_tr.values)
val_ids = set(ids_val.values)
intersection = train_ids.intersection(val_ids)
leakage_checks.append({
    'CHECK': 'Intersecao SK_ID_CURR entre Train e Validation',
    'RESULT': f'{len(intersection)} registros em comum',
    'STATUS': 'FAIL' if len(intersection) > 0 else 'PASS'
})

# 4. Duplicidades entre TRAIN e VALIDATION
n_dup_sk = len(intersection)
leakage_checks.append({
    'CHECK': 'Duplicidades entre TRAIN e VALIDATION',
    'RESULT': f'{n_dup_sk} duplicidades',
    'STATUS': 'FAIL' if n_dup_sk > 0 else 'PASS'
})

# 5. Features com informacao futura conhecida?
# Verificar se ha colunas que podem conter informacao posterior ao TARGET
future_suspects = [c for c in feature_cols if any(x in c.lower() for x in ['future', 'target', 'label', 'outcome'])]
leakage_checks.append({
    'CHECK': 'Features com possivel informacao futura',
    'RESULT': f'{len(future_suspects)} features suspeitas: {future_suspects}' if future_suspects else 'Nenhuma feature suspeita',
    'STATUS': 'WARNING' if future_suspects else 'PASS'
})

# 6. Verificar random_state
leakage_checks.append({
    'CHECK': 'Random state deterministico',
    'RESULT': f'random_state={RANDOM_STATE}',
    'STATUS': 'PASS'
})

# Exibir resultados
leakage_df = pd.DataFrame(leakage_checks)
print("\n=== Data Leakage Check ===")
print(leakage_df.to_string(index=False))

# Resumo
n_pass = (leakage_df['STATUS'] == 'PASS').sum()
n_warn = (leakage_df['STATUS'] == 'WARNING').sum()
n_fail = (leakage_df['STATUS'] == 'FAIL').sum()
overall_status = 'FAIL' if n_fail > 0 else ('WARNING' if n_warn > 0 else 'PASS')
print(f"\nResumo: {n_pass} PASS | {n_warn} WARNING | {n_fail} FAIL")
print(f"Status geral: {overall_status}")
print("\nData leakage check concluido!")

In [0]:
# ============================================================
# SECAO 17 — TABELA FINAL DE AVALIACAO
# ============================================================
print("=" * 60)
print("SECAO 17 — Tabela Final de Avaliacao")
print("=" * 60)

# Schema ja garantido pelo notebook 16
print("Schema credit_risk.analytics garantido.")

# Preparar tabela de resultados de avaliacao
eval_table = metrics_df.copy()
eval_table['VALIDATION_ROWS'] = len(y_val)
eval_table['RANDOM_STATE'] = RANDOM_STATE
eval_table['EVALUATION_DATE'] = pd.Timestamp(datetime.now())

# Reordenar colunas
eval_cols = ['MODEL', 'ROC_AUC', 'PR_AUC', 'ACCURACY', 'PRECISION', 'RECALL', 'F1',
            'SPECIFICITY', 'BALANCED_ACCURACY', 'BRIER_SCORE', 'VALIDATION_ROWS',
            'RANDOM_STATE', 'EVALUATION_DATE']
eval_table = eval_table[eval_cols]

print("\n=== evaluation_results ===")
print(eval_table.to_string(index=False))

# Salvar como tabela Delta
eval_spark_df = spark.createDataFrame(eval_table)
eval_spark_df.write.mode("append").format("delta").saveAsTable(f"{ANALYTICS_SCHEMA}.evaluation_results")
print(f"\nTabela {ANALYTICS_SCHEMA}.evaluation_results criada com {len(eval_table)} registros (append).")

In [0]:
# ============================================================
# SECAO 18 — THRESHOLD TABLE
# ============================================================
print("=" * 60)
print("SECAO 18 — Threshold Table")
print("=" * 60)

# Salvar tabela de analise de threshold
threshold_spark_df = spark.createDataFrame(threshold_df)
threshold_spark_df.write.mode("append").format("delta").saveAsTable(f"{ANALYTICS_SCHEMA}.threshold_analysis")
print(f"Tabela {ANALYTICS_SCHEMA}.threshold_analysis criada com {len(threshold_df)} registros (append).")
print(f"Modelos: {threshold_df['MODEL'].nunique()}")
print(f"Thresholds: {threshold_df['THRESHOLD'].nunique()}")

In [0]:
# ============================================================
# SECAO 19 — CONFUSION MATRIX TABLE
# ============================================================
print("=" * 60)
print("SECAO 19 — Confusion Matrix Table")
print("=" * 60)

confusion_spark_df = spark.createDataFrame(confusion_df)
confusion_spark_df.write.mode("append").format("delta").saveAsTable(f"{ANALYTICS_SCHEMA}.confusion_matrix")
print(f"Tabela {ANALYTICS_SCHEMA}.confusion_matrix criada com {len(confusion_df)} registros (append).")
print(confusion_df.to_string(index=False))

In [0]:
# ============================================================
# SECAO 20 — CALIBRATION TABLE
# ============================================================
print("=" * 60)
print("SECAO 20 — Calibration Table")
print("=" * 60)

calibration_spark_df = spark.createDataFrame(calibration_df)
calibration_spark_df.write.mode("append").format("delta").saveAsTable(f"{ANALYTICS_SCHEMA}.calibration_results")
print(f"Tabela {ANALYTICS_SCHEMA}.calibration_results criada com {len(calibration_df)} registros (append).")
print(calibration_df.head(10).to_string(index=False))

In [0]:
# ============================================================
# SECAO 21 — AUDITORIA
# ============================================================
print("=" * 60)
print("SECAO 21 — Auditoria")
print("=" * 60)

END_TIME = time.time()
END_TIMESTAMP = datetime.now()
DURATION = END_TIME - START_TIME

# Determinar status final
status = "SUCCESS"
if n_fail > 0:
    status = "FAIL"
elif n_warn > 0:
    status = "WARNING"

# Criar registro de auditoria
audit_record = pd.DataFrame([{
    'execution_id': EXECUTION_ID,
    'execution_timestamp': START_TIMESTAMP,
    'notebook': '17_evaluation',
    'status': status,
    'validation_rows': len(y_val),
    'models_evaluated': len(trained_models),
    'features_count': len(feature_cols),
    'random_state': RANDOM_STATE,
    'execution_time_seconds': round(DURATION, 2),
    'evaluation_date': pd.Timestamp(END_TIMESTAMP),
    'notes': f'Leakage check: {n_pass} PASS, {n_warn} WARNING, {n_fail} FAIL. Overall: {overall_status}'
}])

print("\n--- Registro de Auditoria ---")
print(audit_record.to_string(index=False))

# Salvar como tabela Delta append-only
audit_spark_df = spark.createDataFrame(audit_record)
audit_spark_df.write.mode("append").format("delta").saveAsTable(f"{ANALYTICS_SCHEMA}.audit_evaluation")
print(f"\nTabela {ANALYTICS_SCHEMA}.audit_evaluation: registro inserido (append-only)")
print(f"Durecao total: {DURATION:.1f}s ({DURATION/60:.1f} min)")

In [0]:
# ============================================================
# SECAO 22 — RESUMO EXECUTIVO
# ============================================================
print("=" * 60)
print("17_EVALUATION - SUMMARY")
print("=" * 60)

print(f"\nDataset:")
print(f"  Validation rows: {len(y_val):,}")
print(f"  Features: {len(feature_cols)}")
print(f"  Target distribution: {target_dist[1]:,} ({target_dist[1]/len(df_train)*100:.2f}%)")
print(f"  Class imbalance: {class_imbalance:.2f}:1")

print(f"\nModels evaluated:")
for name in trained_models.keys():
    print(f"  - {name}")

print(f"\nMetrics generated:")
print("  - ROC-AUC")
print("  - PR-AUC")
print("  - Precision")
print("  - Recall")
print("  - F1")
print("  - Specificity")
print("  - Balanced Accuracy")
print("  - Brier Score")

print(f"\nThresholds evaluated:")
print(f"  {thresholds_list[0]:.2f} -> {thresholds_list[-1]:.2f} ({len(thresholds_list)} thresholds)")

print(f"\nCalibration:")
print("  Evaluated (Brier Score, Calibration Curve)")

print(f"\nLeakage checks:")
print(f"  {n_pass} PASS | {n_warn} WARNING | {n_fail} FAIL")
print(f"  Overall: {overall_status}")

print(f"\nTables created:")
print(f"  credit_risk.analytics.evaluation_results")
print(f"  credit_risk.analytics.threshold_analysis")
print(f"  credit_risk.analytics.confusion_matrix")
print(f"  credit_risk.analytics.calibration_results")
print(f"  credit_risk.analytics.audit_evaluation")

print(f"\nExecution ID: {EXECUTION_ID}")
print(f"Execution time: {DURATION:.1f}s ({DURATION/60:.1f} min)")

print(f"\nStatus:")
print(f"  {status}")

print(f"\n{'=' * 60}")